# CP4 - Case iFood

Cognitive Data Science + Machine Learning & Modelling

Execute as células nesta ordem. A senha Oracle nunca deve ser escrita no notebook.

In [ ]:
!pip -q install oracledb scikit-learn pandas
import getpass
import oracledb
import pandas as pd

## 1. Ler a tabela do Oracle

Antes desta célula, crie e carregue `IFOOD_CUSTOMERS` no SQL Developer. No Colab, abra o painel de chave `Secrets`, crie `ORACLE_PASSWORD`, informe a senha e ative o acesso para este notebook. Se o segredo não estiver configurado, a célula solicitará a senha sem exibi-la.

In [ ]:
ORACLE_USER = 'RM573854'
try:
    from google.colab import userdata
    ORACLE_PASSWORD = userdata.get('ORACLE_PASSWORD')
except Exception:
    ORACLE_PASSWORD = None
if not ORACLE_PASSWORD:
    ORACLE_PASSWORD = getpass.getpass('Senha Oracle (não exibida): ')
ORACLE_DSN = oracledb.makedsn('oracle.fiap.com.br', 1521, service_name='orcl')

SQL = '''SELECT
    ID, YEAR_BIRTH, EDUCATION, MARITAL_STATUS, INCOME, KIDHOME, TEENHOME,
    DT_CUSTOMER, RECENCY, MNTWINES, MNTFRUITS, MNTMEATPRODUCTS,
    MNTFISHPRODUCTS, MNTSWEETPRODUCTS, MNTGOLDPRODS, NUMDEALSPURCHASES,
    NUMWEBPURCHASES, NUMCATALOGPURCHASES, NUMSTOREPURCHASES,
    NUMWEBVISITSMONTH, ACCEPTEDCMP3, ACCEPTEDCMP4, ACCEPTEDCMP5,
    ACCEPTEDCMP1, ACCEPTEDCMP2, COMPLAIN, Z_COSTCONTACT, Z_REVENUE, RESPONSE
FROM IFOOD_CUSTOMERS
ORDER BY ID'''

with oracledb.connect(user=ORACLE_USER, password=ORACLE_PASSWORD, dsn=ORACLE_DSN) as connection:
    with connection.cursor() as cursor:
        cursor.execute(SQL)
        columns = [item[0] for item in cursor.description]
        df = pd.DataFrame(cursor.fetchall(), columns=columns)

print('shape:', df.shape)
print('INCOME nulo:', int(df['INCOME'].isna().sum()))

## 2. Pipeline e Feature Engineering

O holdout estratificado é separado antes do tuning. Imputação, codificação e treino usam somente o conjunto de treino.

In [ ]:
"""Pipeline reproduzivel da Parte 2 do CP4.

Pode receber o DataFrame retornado pelo Oracle ou ler o CSV para teste local.
O holdout fica separado antes do tuning. Imputacao e one-hot ficam dentro do
Pipeline, portanto usam apenas os dados de treino durante o ajuste.
"""

from pathlib import Path

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier


RANDOM_STATE = 42
DATA_PATH = Path("data.csv")


def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    result = df.copy()
    result.columns = [str(column).upper() for column in result.columns]
    return result


def build_features(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series]:
    data = normalize_columns(df)
    data["DT_CUSTOMER"] = pd.to_datetime(data["DT_CUSTOMER"], errors="coerce")

    # Features criadas antes do split, sem usar RESPONSE.
    data["AGE_AT_2014"] = 2014 - data["YEAR_BIRTH"]
    data["TOTAL_CHILDREN"] = data["KIDHOME"] + data["TEENHOME"]
    spend_columns = [
        "MNTWINES", "MNTFRUITS", "MNTMEATPRODUCTS",
        "MNTFISHPRODUCTS", "MNTSWEETPRODUCTS", "MNTGOLDPRODS",
    ]
    purchase_columns = [
        "NUMDEALSPURCHASES", "NUMWEBPURCHASES",
        "NUMCATALOGPURCHASES", "NUMSTOREPURCHASES",
    ]
    campaign_columns = [
        "ACCEPTEDCMP1", "ACCEPTEDCMP2", "ACCEPTEDCMP3",
        "ACCEPTEDCMP4", "ACCEPTEDCMP5",
    ]
    data["TOTAL_SPEND"] = data[spend_columns].sum(axis=1)
    data["TOTAL_PURCHASES"] = data[purchase_columns].sum(axis=1)
    data["CAMPAIGNS_ACCEPTED"] = data[campaign_columns].sum(axis=1)
    data["WEB_PURCHASE_SHARE"] = (
        data["NUMWEBPURCHASES"] / data["TOTAL_PURCHASES"].replace(0, 1)
    )
    data["CUSTOMER_TENURE_DAYS"] = (
        pd.Timestamp("2014-06-30") - data["DT_CUSTOMER"]
    ).dt.days

    y = data.pop("RESPONSE").astype(int)
    X = data.drop(
        columns=["ID", "YEAR_BIRTH", "DT_CUSTOMER", "Z_COSTCONTACT", "Z_REVENUE"]
    )
    return X, y


def make_preprocessor(X: pd.DataFrame) -> ColumnTransformer:
    numeric_columns = X.select_dtypes(include="number").columns.tolist()
    categorical_columns = X.select_dtypes(exclude="number").columns.tolist()

    numeric = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
    ])
    categorical = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])
    return ColumnTransformer([
        ("numeric", numeric, numeric_columns),
        ("categorical", categorical, categorical_columns),
    ])


def evaluate(name: str, model: Pipeline, X_test: pd.DataFrame, y_test: pd.Series) -> dict:
    predictions = model.predict(X_test)
    probabilities = model.predict_proba(X_test)[:, 1]
    metrics = {
        "model": name,
        "precision": precision_score(y_test, predictions, zero_division=0),
        "recall": recall_score(y_test, predictions, zero_division=0),
        "f1": f1_score(y_test, predictions, zero_division=0),
        "roc_auc": roc_auc_score(y_test, probabilities),
    }
    print(pd.Series(metrics).to_string())
    return metrics


def run(df: pd.DataFrame) -> pd.DataFrame:
    X, y = build_features(df)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
    )

    reference = Pipeline([
        ("preprocessor", make_preprocessor(X_train)),
        ("model", DecisionTreeClassifier(random_state=RANDOM_STATE)),
    ])
    reference.fit(X_train, y_train)
    results = [evaluate("reference", reference, X_test, y_test)]

    tuned = Pipeline([
        ("preprocessor", make_preprocessor(X_train)),
        ("model", DecisionTreeClassifier(random_state=RANDOM_STATE)),
    ])
    search = GridSearchCV(
        tuned,
        param_grid={
            "model__criterion": ["gini", "entropy", "log_loss"],
            "model__max_depth": [3, 5, 8, None],
            "model__min_samples_leaf": [1, 5, 10, 20],
            "model__class_weight": [None, "balanced"],
        },
        scoring="roc_auc",
        cv=5,
        n_jobs=-1,
        refit=True,
    )
    search.fit(X_train, y_train)
    print("best_params")
    print(search.best_params_)
    results.append(evaluate("tuned", search.best_estimator_, X_test, y_test))
    return pd.DataFrame(results)




In [4]:
metrics = run(df)
metrics

model        reference
precision     0.563636
recall        0.462687
f1            0.508197
roc_auc       0.697027
best_params
{'model__class_weight': 'balanced', 'model__criterion': 'gini', 'model__max_depth': 8, 'model__min_samples_leaf': 20}
model           tuned
precision    0.362319
recall       0.746269
f1           0.487805
roc_auc      0.834019

## 3. Resultado

Na execução validada com o DataFrame do Oracle, o ROC AUC passou de 0,6970 para 0,8340. A referência tem precision 0,5636, recall 0,4627 e F1 0,5082. O modelo ajustado tem precision 0,3623, recall 0,7463 e F1 0,4878. Confirme a contagem de 2.240 linhas e 29 colunas antes de interpretar as métricas. Use `File > Download > Download .ipynb` para baixar o notebook com as saídas.